<a href="https://colab.research.google.com/github/buildwithdemis/machinelearning/blob/main/Winnipeg_Transit_On_Time_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Business problem:

Public transit reliability is an important factor in how citizens experience their daily commute. In Winnipeg, buses occasionally arrive either ahead of schedule, right on time, or later than planned. This unpredictability can cause inconvenience for riders, impact connections between routes, and reduce overall confidence in the transit system.

Our goal in this project is to explore whether we can use historical transit schedule and performance data to predict the punctuality of buses. By looking at patterns such as the route number, stop location, time of day, and day of week, we aim to forecast whether a bus is likely to be early, on time, or late.

# Type of ML problem:
   ## Classification (Early, On-time vs Late).
   * This is a classification problem in machine learning, since we are assigning each bus trip to one of three categories: Early, On-time, or Late.

The overall objective is to train and evaluate a predictive model within Azure Machine Learning, and then log and register the model using MLflow so that it can be managed and potentially deployed as part of a larger intelligent transit system.

# Goal:
   Train, evaluate, and register a model in Azure ML.

# Dataset Selection

we are using the Winnipeg Transit On-Time Performance dataset from the open data portal.
* source: https://data.winnipeg.ca/Transit/Recent-Transit-On-Time-Performance-Data/gp3k-am4u/about_data

* Date range: May 17,2025 to Aug. 19, 2025
* since the whole data size is larger than 600Mb, i just take the first 100k rows for the purpose of demo

# Load the data

In [1]:
!wget https://raw.githubusercontent.com/buildwithdemis/machinelearning/refs/heads/main/al/Transit_On-Time_Performance_100k.csv

--2025-08-20 03:56:24--  https://raw.githubusercontent.com/buildwithdemis/machinelearning/refs/heads/main/al/Transit_On-Time_Performance_100k.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13206334 (13M) [text/plain]
Saving to: ‘Transit_On-Time_Performance_100k.csv’

Transit_On-Time_Per 100%[===================>]  12.59M  --.-KB/s    in 0.1s    

2025-08-20 03:56:25 (110 MB/s) - ‘Transit_On-Time_Performance_100k.csv’ saved [13206334/13206334]



In [11]:
import pandas
# Read the text file containing data using pandas
df = pandas.read_csv('Transit_On-Time_Performance_100k.csv', delimiter=',')


print("number of rows: " , df.size)

# Because there are a lot of data, use head() to only print the first few rows
df.head(10)

number of rows:  900000


,Row ID,Stop Number,Route Number,Route Name,Route Destination,Day Type,Scheduled Time,Deviation,Location
0,1529721475,40198,D17,Talbot - Selkirk,Kildonan Place,Saturday,2025 Jul 05 08:28:45 AM,190,POINT (-97.0636837759858 49.8993976819722)
1,1529707518,31013,39,Inkster,Waterford Green,Saturday,2025 Jul 05 10:07:13 PM,-57,POINT (-97.1300872125569 49.9294967207071)
2,1529682770,10339,43,Watt - Logan,Gateway,Saturday,2025 Jul 05 11:58:02 AM,173,POINT (-97.1504360488864 49.9071647664343)
3,1529693051,30165,38,Mountain - Munroe,Kildonan Place,Saturday,2025 Jul 05 04:31:19 PM,-130,POINT (-97.1361307804554 49.9216514650652)
4,1529696396,60450,70,Roblin,Unicity,Saturday,2025 Jul 05 08:32:43 PM,48,POINT (-97.2036614122655 49.8747203214952)
5,1529612457,60064,91,St. Norbert,St. Norbert,Saturday,2025 Jul 05 07:13:00 AM,-192,POINT (-97.1565841949361 49.7896341634893)
6,1529696447,60556,70,Roblin,Unicity,Saturday,2025 Jul 05 08:51:48 PM,64,POINT (-97.2924548225466 49.8594937363161)
7,1529696398,60444,70,Roblin,Unicity,Saturday,2025 Jul 05 08:33:35 PM,69,POINT (-97.2054612101309 49.8715999527232)
8,1529696520,60461,70,Roblin,Polo Park,Saturday,2025 Jul 05 09:41:56 PM,166,POINT (-97.2136300871861 49.8663381400167)
9,1529612543,60026,91,St. Norbert,St. Norbert,Saturday,2025 Jul 05 07:52:42 AM,-49,POINT (-97.1569697482449 49.7794757683298)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Row ID             100000 non-null  int64 
 1   Stop Number        100000 non-null  int64 
 2   Route Number       100000 non-null  object
 3   Route Name         100000 non-null  object
 4   Route Destination  100000 non-null  object
 5   Day Type           100000 non-null  object
 6   Scheduled Time     100000 non-null  object
 7   Deviation          100000 non-null  object
 8   Location           100000 non-null  object
dtypes: int64(2), object(7)
memory usage: 6.9+ MB


## Description of each column

* Stop Number (int64): Numeric ID of the bus stop where the observation was recorded. Each stop number uniquely identifies a physical bus stop in Winnipeg.
* Route Number (object): The official route number of the bus (e.g., 16, 21, 60, D13,F5). Routes with higher traffic or longer distances may show different on-time patterns.

* Route Name (object): Human-readable name of the bus route (e.g., “GRANT EXPRESS”). Often correlated with route number but provides more descriptive context.
* Route Destination (object): The end destination of the route for that trip (e.g., “Downtown”, “Kildonan Place”). Useful to distinguish between directions of travel on the same route.
* Day Type (object): Indicates the type of service day (e.g., “Weekday”, “Saturday”, “Sunday/Holiday”). Bus schedules and traffic patterns differ significantly by day type.

* Scheduled Time (object): The scheduled departure or arrival time at the bus stop (e.g., “08:45:00”).Needs to be converted into a time-based feature (hour of day, morning/evening, peak vs off-peak).
* Location (object): The geographic description of the stop location (e.g., “Portage & Main”).Can be used for spatial analysis or grouped by high-traffic areas.

* Deviation (object): The difference between actual arrival time and scheduled time. Usually expressed in minutes (negative = early, zero = on time, positive = late).

This is the target variable, which we will classify into categories (Early, On-time, Late).

Label = Late if offset > 60 sec

Label = On-time if offset between 0-60 sec

Label = EARLY if offset ≤ 0 sec